# Hypothesis Testing

## 1. Introduction
In the previous Exploratory Data Analysis (EDA) stage, I visually observed potential relationships between socio-economic factors and traffic death rates. In this notebook, I will use statistical hypothesis testing to strictly prove or disprove assumptions. I will conduct three different statistical tests covering the economic, behavioral, and temporal dimensions of the dataset.

In [32]:
# importing libraries.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# setting the visual style
sns.set_theme(style="whitegrid", palette="muted")

# loading the processed dataset
df = pd.read_csv('processed_dataset.csv')
print(f"Data loaded successfully. Shape: {df.shape}")

Data loaded successfully. Shape: (3677, 7)


## 2. Hypothesis 1: Economic Impact (Independent T-Test)
Our EDA boxplots suggested that lower-income countries suffer from higher traffic death rates. To prove this, we will divide the world into two halves based on the median GDP and run an Independent T-Test (Welch's T-Test, assuming unequal variances).

* **H0 (Null Hypothesis):** There is no significant difference in the mean traffic death rates between High-GDP and Low-GDP countries.
* **H1 (Alternative Hypothesis):** The mean traffic death rate in Low-GDP countries is significantly different (higher) than in High-GDP countries.
* **Significance Level ($\alpha$):** 0.05

In [33]:
# find the median GDP to split the data.
median_gdp = df['GDP_Per_Capita'].median()

# split the data into two independent groups.
low_gdp_group = df[df['GDP_Per_Capita'] < median_gdp]['Death_Rate_Per_100k']
high_gdp_group = df[df['GDP_Per_Capita'] >= median_gdp]['Death_Rate_Per_100k']

# Performing Welch's T-Test
t_stat, p_value = stats.ttest_ind(low_gdp_group, high_gdp_group, equal_var=False)

# printing the results.
print("--- INDEPENDENT T-TEST RESULTS ---")
print(f"Mean Death Rate (Low GDP):  {low_gdp_group.mean():.2f}")
print(f"Mean Death Rate (High GDP): {high_gdp_group.mean():.2f}")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value:     {p_value:.4e}")

# displaying the end result objectively.
if p_value < 0.05:
    print("\nConclusion: REJECT H0.")
    print("There is a statistically significant difference in the mean traffic death rates between the two income groups.")
else:
    print("\nConclusion: FAIL TO REJECT H0.")
    print("There is no statistically significant difference in the mean traffic death rates between the two income groups.")

--- INDEPENDENT T-TEST RESULTS ---
Mean Death Rate (Low GDP):  19.32
Mean Death Rate (High GDP): 15.51
T-Statistic: 10.8932
P-Value:     3.2684e-27

Conclusion: REJECT H0.
There is a statistically significant difference in the mean traffic death rates between the two income groups.


## 3. Hypothesis 2: Behavioral Impact (Pearson Correlation)
 Starting the behavioral analysis by examining the linear relationship between alcohol consumption and traffic death rates using the **Pearson Correlation Coefficient**. This test measures the strength and direction of a linear trend between two continuous variables.

* **H0 (Null Hypothesis):** There is no linear correlation between Alcohol Consumption and Traffic Death Rates ($r = 0$).
* **H1 (Alternative Hypothesis):** There is a significant linear correlation between Alcohol Consumption and Traffic Death Rates ($r \neq 0$).

In [34]:
# extracting the variables
alcohol = df['Alcohol_Per_Capita']
deaths = df['Death_Rate_Per_100k']

# performing the Pearson Correlation Test.
pearson_r, pearson_p = stats.pearsonr(alcohol, deaths)

print("--- PEARSON CORRELATION RESULTS ---")
print(f"Correlation Coefficient (r): {pearson_r:.4f}")
print(f"P-Value: {pearson_p:.4e}")

# displaying results.
if pearson_p < 0.05:
    print("\nConclusion: REJECT H0.")
    print("There is a statistically significant linear correlation between the two variables.")
else:
    print("\nConclusion: FAIL TO REJECT H0.")
    print("There is no statistically significant linear correlation between the two variables.")

--- PEARSON CORRELATION RESULTS ---
Correlation Coefficient (r): -0.2458
P-Value: 9.4383e-52

Conclusion: REJECT H0.
There is a statistically significant linear correlation between the two variables.


## 3.1. Robustness Check: Spearman Rank Correlation
As identified in the EDA part, the variables exhibit significant **right-skewness**. Since Pearson is sensitive to outliers and non-normal distributions, performing a **Spearman Rank Correlation** as a robustness check. Spearman evaluates the monotonic relationship using the ranks of the data, making it far more reliable for skewed panel data.



In [35]:
# performing the Spearman Rank Correlation Test.
spearman_rho, spearman_p = stats.spearmanr(alcohol, deaths)

print("--- SPEARMAN RANK CORRELATION RESULTS ---")
print(f"Spearman's Rho (ρ): {spearman_rho:.4f}")
print(f"P-Value: {spearman_p:.4e}")

# displaying results.
if spearman_p < 0.05:
    print("\nConclusion: REJECT H0.")
    print("There is a statistically significant monotonic correlation between the two variables.")
else:
    print("\nConclusion: FAIL TO REJECT H0.")
    print("There is no statistically significant monotonic correlation between the two variables.")

--- SPEARMAN RANK CORRELATION RESULTS ---
Spearman's Rho (ρ): -0.2469
P-Value: 3.2665e-52

Conclusion: REJECT H0.
There is a statistically significant monotonic correlation between the two variables.


## 4. Hypothesis 3: Temporal Evolution (Paired T-Test)
Did the world get safer or more dangerous over the 20 year timeline? To answer this, I will compare the death rates of the *exact same countries* in the year 2000 versus the year 2019. Since we are comparing the same entities at two different times, a Paired T-Test is the perfect choice.

* **H0 (Null Hypothesis):** The mean difference in traffic death rates between 2000 and 2019 is zero (No improvement).
* **H1 (Alternative Hypothesis):** The mean difference in traffic death rates between 2000 and 2019 is not zero (Global roads became safer or more dangerous).
* **Significance Level ($\alpha$):** 0.05

In [36]:
# filtering the data for the starting year (2000) and ending year (2019).
df_2000 = df[df['Year'] == 2000][['Country_Code', 'Death_Rate_Per_100k']].rename(columns={'Death_Rate_Per_100k': 'Rate_2000'})
df_2019 = df[df['Year'] == 2019][['Country_Code', 'Death_Rate_Per_100k']].rename(columns={'Death_Rate_Per_100k': 'Rate_2019'})

# merging to ensure that only test countries that have data in BOTH years (Paired samples).
df_paired = pd.merge(df_2000, df_2019, on='Country_Code', how='inner')

# performing the Paired T-Test.
t_stat_paired, p_value_paired = stats.ttest_rel(df_paired['Rate_2000'], df_paired['Rate_2019'])

print("--- PAIRED T-TEST RESULTS (2000 vs 2019) ---")
print(f"Number of matched countries: {len(df_paired)}")
print(f"Global Mean Death Rate in 2000: {df_paired['Rate_2000'].mean():.2f}")
print(f"Global Mean Death Rate in 2019: {df_paired['Rate_2019'].mean():.2f}")
print(f"T-Statistic: {t_stat_paired:.4f}")
print(f"P-Value:     {p_value_paired:.4e}")

# displaying the results.
if p_value_paired < 0.05:
    print("\nConclusion: REJECT H0.")
    print("There is a statistically significant difference in the global average traffic death rate between 2000 and 2019.")
else:
    print("\nConclusion: FAIL TO REJECT H0.")
    print("There is no statistically significant difference between the two years.")

--- PAIRED T-TEST RESULTS (2000 vs 2019) ---
Number of matched countries: 182
Global Mean Death Rate in 2000: 19.59
Global Mean Death Rate in 2019: 15.08
T-Statistic: 10.1164
P-Value:     2.3726e-19

Conclusion: REJECT H0.
There is a statistically significant difference in the global average traffic death rate between 2000 and 2019.


## 5. Final Conclusion

In this project, the statistical tests confirmed the patterns that were found during the EDA phase. Here are the main conclusions:

**1. Wealthier Countries Have Safer Roads (Welch's T-Test):**
The test showed a clear difference (with a p-value near zero). Low-income countries have a much higher average traffic death rate (19.32) compared to high-income countries (15.51). A better economy likely means better road infrastructure, safer cars, and better hospitals.

**2. The Alcohol Issue - Correlation is not Causation (Pearson & Spearman):** Found a significant but a *Negative* correlation between alcohol and traffic deaths. Does this mean drinking alcohol makes roads safer? Most probably no. This happens because of a **confounding variable**: High-income countries (like in Europe) consume more alcohol, but they also have much safer roads. The wealth factor hides the real danger of alcohol when looking at the global scale.

**3. The World is Getting Safer Over Time (Paired T-Test):**
When we compared the same countries in 2000 and 2019, the global average death rate dropped significantly from 19.59 to 15.08. This shows that global safety efforts, such as seatbelt laws and better car designs, have successfully saved lives over the past 20 years.

---
**Final Note:** This project shows that traffic fatalities are not just random accidents. They are closely tied to country's attributes such as economic level and improve as technology and infrastructure develop.